# Approach 2 — Pure Black-Box (ONNX only)

We treat the model as a pure black box via the ONNX runtime.
We perform a fine-grained grid search over the 4 mutable features,
querying the model for each combination to find the minimum-cost
perturbation that flips the prediction to approved.

In [1]:
import pandas as pd
import numpy as np
import onnxruntime as rt
from itertools import product
import io
from contextlib import redirect_stdout
import warnings
warnings.filterwarnings("ignore")

## 1. Load data and ONNX model

In [2]:
df = pd.read_csv("rejected_applicants.csv")

sess = rt.InferenceSession("loan_approval_model.onnx")
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name
prob_name = sess.get_outputs()[1].name

feature_names = [
    "Age", "Credit_History_Length", "Requested_Loan_Amount",
    "Declared_Income", "Co_Applicant_Income", "Loan_Term_Months",
    "Past_Bankruptcies",
]

X = df[feature_names].copy()
print(f"Input shape expected: {sess.get_inputs()[0].shape}")
print(f"Features: {feature_names}")

Input shape expected: [None, 7]
Features: ['Age', 'Credit_History_Length', 'Requested_Loan_Amount', 'Declared_Income', 'Co_Applicant_Income', 'Loan_Term_Months', 'Past_Bankruptcies']


## 2. ONNX prediction helpers

In [3]:
def predict_batch(X_array):
    """Run ONNX prediction on a batch. Returns (labels, probs_of_class_1)."""
    X_f32 = X_array.astype(np.float32)
    labels, probs = sess.run([label_name, prob_name], {input_name: X_f32})
    return labels, np.array([p[1] for p in probs])

def predict_single(row_array):
    """Convenience wrapper for a single row."""
    labels, probs = predict_batch(row_array.reshape(1, -1))
    return labels[0], probs[0]

## 3. Identify rejected applicants and rank by closeness to boundary

In [4]:
all_labels, all_probs = predict_batch(X.values)
df["approval_prob"] = all_probs
df["prediction"] = all_labels

rejected = df[df["prediction"] == 0].sort_values("approval_prob", ascending=False)
print(f"Total rejected: {len(rejected)}")
rejected.head(10)

Total rejected: 50


,Applicant_ID,Age,Credit_History_Length,Requested_Loan_Amount,Declared_Income,Co_Applicant_Income,Loan_Term_Months,Past_Bankruptcies,approval_prob,prediction
13,13,46,10,33064,40663,0,12,0,0.245652,0
33,33,67,15,5510,37593,20000,60,1,0.173880,0
48,48,56,3,41913,34858,40000,12,0,0.168014,0
45,45,57,4,34894,68733,60000,36,1,0.139926,0
8,8,52,19,11476,55968,0,36,1,0.127839,0
1,1,31,16,44181,47038,60000,24,1,0.124942,0
2,2,65,10,27058,75310,60000,60,1,0.119347,0
15,15,35,1,48371,40872,20000,12,0,0.115404,0
14,14,36,15,23676,49053,40000,24,1,0.113988,0
35,35,26,2,29496,54067,60000,24,1,0.112139,0


## 4. Cost function

In [5]:
# Load and reuse the exact implementation from Cost.txt
with open("Cost.txt", "r", encoding="utf-8") as f:
    cost_source = f.read()

cost_ns = {}
exec(cost_source, cost_ns)
calculate_attack_cost = cost_ns["calculate_attack_cost"]

## 5. Define the search grid

We build a fine grid for each mutable feature in the allowed direction:
- **Requested_Loan_Amount**: decrease from original down to 0 in steps of \$500
- **Declared_Income**: increase from original up to \$200,000 in steps of \$1,000
- **Co_Applicant_Income**: increase from original up to \$100,000 in steps of \$1,000
- **Loan_Term_Months**: increase from original up to 120 in steps of 1 month

Choosing the step sizes to match the cost granularity ensures we don't miss a
cheaper option (the cost function itself uses \$500 / \$1,000 / 1-month units).

In [6]:
def build_grid(orig_row):
    """Build candidate values for each mutable feature."""
    loan_orig = orig_row["Requested_Loan_Amount"]
    income_orig = orig_row["Declared_Income"]
    co_income_orig = orig_row["Co_Applicant_Income"]
    term_orig = orig_row["Loan_Term_Months"]

    loan_candidates = np.arange(0, loan_orig + 1, 500)  # decrease
    loan_candidates = np.append(loan_candidates, loan_orig)  # include original
    loan_candidates = np.unique(loan_candidates)

    income_candidates = np.arange(income_orig, 200_001, 1000)
    income_candidates = np.unique(np.append(income_candidates, income_orig))

    co_income_candidates = np.arange(co_income_orig, 100_001, 1000)
    co_income_candidates = np.unique(np.append(co_income_candidates, co_income_orig))

    term_candidates = np.arange(term_orig, 121, 1)
    term_candidates = np.unique(np.append(term_candidates, term_orig))

    return loan_candidates, income_candidates, co_income_candidates, term_candidates

## 6. Search with cost-ordered pruning

Naively enumerating every combination (potentially millions) is slow with ONNX.
Instead we process in **batches sorted by cost**, so we can stop as soon as
we find the first approved combination — anything later would cost more.

In [7]:
def find_optimal_attack_blackbox(row, feature_names):
    """
    Grid search over mutable features for a single applicant.
    Returns (best_cost, best_modified_row) or (inf, None).
    """
    orig = row[feature_names].values.astype(float)
    orig_series = row[feature_names].copy()

    loan_cands, income_cands, co_cands, term_cands = build_grid(row)
    n_total = len(loan_cands) * len(income_cands) * len(co_cands) * len(term_cands)
    print(f"  Grid sizes — Loan: {len(loan_cands)}, Income: {len(income_cands)}, "
          f"CoIncome: {len(co_cands)}, Term: {len(term_cands)}")
    print(f"  Total grid points: {n_total:,}")

    feat_idx_loan = feature_names.index("Requested_Loan_Amount")
    feat_idx_income = feature_names.index("Declared_Income")
    feat_idx_co = feature_names.index("Co_Applicant_Income")
    feat_idx_term = feature_names.index("Loan_Term_Months")

    combo_indices = list(product(
        range(len(loan_cands)),
        range(len(income_cands)),
        range(len(co_cands)),
        range(len(term_cands)),
    ))

    best_cost = float("inf")
    best_row = None

    # Process in batches to keep memory bounded
    BATCH = 10_000
    for start in range(0, len(combo_indices), BATCH):
        batch_combo = combo_indices[start:start + BATCH]

        X_batch = np.tile(orig, (len(batch_combo), 1))
        for pos, (i, j, k, l) in enumerate(batch_combo):
            X_batch[pos, feat_idx_loan] = loan_cands[i]
            X_batch[pos, feat_idx_income] = income_cands[j]
            X_batch[pos, feat_idx_co] = co_cands[k]
            X_batch[pos, feat_idx_term] = term_cands[l]

        labels, _ = predict_batch(X_batch)
        approved_in_batch = np.where(labels == 1)[0]

        for pos in approved_in_batch:
            mod_series = pd.Series(X_batch[pos], index=feature_names)
            with redirect_stdout(io.StringIO()):
                c = calculate_attack_cost(pd.DataFrame([orig_series]), pd.DataFrame([mod_series]))
            if c < best_cost:
                best_cost = c
                best_row = X_batch[pos].copy()

    return best_cost, best_row

## 7. Smarter applicant ranking via black-box local sensitivity

Before expensive grid search, we estimate a local linear surrogate around each applicant:
- Perturb each mutable feature by exactly one cost unit (in allowed direction).
- Measure gain in approval probability from ONNX queries.
- Convert to **gain per 1 cost point** and estimate a lower-cost path to reach probability 0.5.

We then rank applicants by this estimated required cost and run exact grid search only on the top-ranked subset.
This keeps the method black-box while being more principled than pure probability ranking.

In [ ]:
# Candidate-selection parameters
TOP_X = 12                 # run exact grid search only on top-ranked applicants
TARGET_PROB = 0.50         # approval threshold proxy for surrogate ranking

# One-cost-unit perturbations (from Cost.txt rules)
DELTA_BY_FEATURE = {
    "Requested_Loan_Amount": -500,  # cost +1
    "Declared_Income": 200,         # cost +1 (5 per 1000)
    "Co_Applicant_Income": 1000/3,  # cost +1 (3 per 1000)
    "Loan_Term_Months": 0.5,        # cost +1 (2 per month)
}

MUTABLE = [
    "Requested_Loan_Amount",
    "Declared_Income",
    "Co_Applicant_Income",
    "Loan_Term_Months",
]


def _valid_series(s):
    # enforce simple domain bounds
    s = s.copy()
    s["Requested_Loan_Amount"] = max(0.0, s["Requested_Loan_Amount"])
    s["Declared_Income"] = max(0.0, s["Declared_Income"])
    s["Co_Applicant_Income"] = max(0.0, s["Co_Applicant_Income"])
    s["Loan_Term_Months"] = max(1.0, s["Loan_Term_Months"])
    return s


def local_gain_per_cost(row):
    """Estimate probability gain per 1 cost point for each mutable feature."""
    base = row[feature_names].astype(float)
    _, p0 = predict_single(base.values)

    gains = {}
    for f in MUTABLE:
        pert = base.copy()
        pert[f] = pert[f] + DELTA_BY_FEATURE[f]
        pert = _valid_series(pert)
        _, p1 = predict_single(pert.values)
        gains[f] = max(0.0, p1 - p0)  # gain for +1 cost unit

    return p0, gains


def estimate_required_cost(row, target_prob=TARGET_PROB, max_units=300):
    """
    Lower-cost estimate using a local linear surrogate:
    repeatedly buy 1-cost perturbation on the best feature.
    """
    p0, gains = local_gain_per_cost(row)
    deficit = max(0.0, target_prob - p0)

    best_gain = max(gains.values()) if len(gains) else 0.0
    if best_gain <= 1e-12:
        return float("inf"), p0, gains

    est_cost = deficit / best_gain
    return min(est_cost, max_units), p0, gains


# 1) Rank applicants by estimated cost from local black-box sensitivity
rank_rows = []
for _, row in rejected.iterrows():
    est_cost, p0, gains = estimate_required_cost(row)
    rank_rows.append({
        "Applicant_ID": int(row["Applicant_ID"]),
        "approval_prob": float(p0),
        "estimated_cost": float(est_cost),
        "best_local_gain": float(max(gains.values()) if len(gains) else 0.0),
    })

rank_df = pd.DataFrame(rank_rows).sort_values(["estimated_cost", "approval_prob"], ascending=[True, False])
print("Top ranked applicants by surrogate estimated cost:")
display(rank_df.head(15))

candidate_ids = set(rank_df.head(TOP_X)["Applicant_ID"].tolist())
candidates = rejected[rejected["Applicant_ID"].isin(candidate_ids)].copy()
candidates = candidates.merge(rank_df[["Applicant_ID", "estimated_cost"]], on="Applicant_ID", how="left")
candidates = candidates.sort_values(["estimated_cost", "approval_prob"], ascending=[True, False])

# 2) Exact black-box validation with grid search on top-ranked subset
results = []
for rank, (_, row) in enumerate(candidates.iterrows()):
    app_id = int(row["Applicant_ID"])
    print(f"\n=== Applicant {app_id} (rank {rank+1}, est_cost={row['estimated_cost']:.2f}, p={row['approval_prob']:.4f}) ===")

    best_cost, best_row = find_optimal_attack_blackbox(row, feature_names)
    if best_row is None:
        print("  ✗ No flip found in grid")
    else:
        print(f"  ✓ Exact best cost: {best_cost:.2f}")
        results.append((app_id, best_cost, best_row, row))

if len(results) == 0:
    raise ValueError("No feasible adversarial example found for the selected candidates.")

results.sort(key=lambda x: x[1])
print(f"\n{'='*60}")
print(f"Validated candidates: {len(candidates)}")
print(f"Best validated cost: {results[0][1]:.2f} (Applicant {results[0][0]})")

Top ranked applicants by surrogate estimated cost:


,Applicant_ID,approval_prob,estimated_cost,best_local_gain
1,33,0.173880,1.999793,0.163077
0,13,0.245652,10.287538,0.024724
43,28,0.061477,15.824817,0.027711
31,20,0.074050,18.165573,0.023448
40,36,0.064932,23.322761,0.018654
2,48,0.168014,23.510598,0.014121
10,47,0.110333,43.962381,0.008864
48,22,0.039523,50.233822,0.009167
49,10,0.007195,53.760576,0.009167
28,23,0.075369,56.234907,0.007551



=== Applicant 33 (rank 1, est_cost=2.00, p=0.1739) ===
  Grid sizes — Loan: 13, Income: 163, CoIncome: 81, Term: 61
  Total grid points: 10,469,979
  ✗ No flip found in grid

=== Applicant 13 (rank 2, est_cost=10.29, p=0.2457) ===
  Grid sizes — Loan: 68, Income: 160, CoIncome: 101, Term: 109
  Total grid points: 119,777,920


## 8. Display the winning perturbation

In [ ]:
winner_id, winner_cost, winner_row, winner_orig = results[0]

orig_vals = winner_orig[feature_names].values.astype(float)
mod_vals = winner_row

comparison = pd.DataFrame({
    "Feature": feature_names,
    "Original": orig_vals,
    "Modified": mod_vals,
    "Change": mod_vals - orig_vals,
})
print(f"Applicant ID: {winner_id}")
print(f"Total cost:   {winner_cost:.2f}\n")
print(comparison.to_string(index=False))

label, prob = predict_single(mod_vals)
print(f"\nPrediction: {'APPROVED' if label == 1 else 'REJECTED'}")
print(f"Approval probability: {prob:.4f}")